# Data Quality — Add Checks Notebook

**Run this notebook to register your semantic models and the DAX expressions you want to validate.**

For each metric you want to track across multiple models:
1. Pick a **check name** — use the same name across all models (e.g., `"Total Sales"`)
2. Add one row per model with the workspace, dataset, and DAX that returns that metric for that model
3. The job will compare all models against the **first one listed** for each check

The table is safe to re-run — duplicate registrations won't create duplicate rows.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
LAKEHOUSE_NAME = "MyLakehouse"    # Your existing Lakehouse name
SCHEMA_NAME    = "data_quality"   # Schema (namespace) where tables were created

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

## How to Add Checks

Edit the `checks` list in the cell below. For each row, provide:

| Column | Example | Options |
|---|---|---|
| **check_name** | `"Total Sales"` | Must be identical across all models for that metric |
| **model_name** | `"Finance GAAP"` | Your label for this model |
| **workspace_name** | `"Finance Workspace"` | Fabric workspace name |
| **dataset_name** | `"Finance Model"` | Semantic model name |
| **dax_expression** | `'EVALUATE ROW("value", [Sales Amount])'` | DAX returning one number |
| **run_frequency** | `"ONCE_PER_DAY"` | `ONCE_PER_DAY` (default, max once daily) or `MULTIPLE_PER_DAY` (can run multiple times/day) |

**run_frequency:** 
- Use `ONCE_PER_DAY` for daily metrics (most common)
- Use `MULTIPLE_PER_DAY` if you need to validate this check multiple times per day

In [ ]:
from pyspark.sql.functions import lit

# ── EDIT THE LIST BELOW ──────────────────────────────────────────────────────
#
#   (check_name,        model_name,      workspace_name,        dataset_name,         dax_expression,                                    run_frequency)
#
checks = [
    ("Total Sales",     "Finance",       "Finance Workspace",   "Finance Model",      'EVALUATE ROW("value", [Sales Amount])',          "ONCE_PER_DAY"),
    ("Total Sales",     "Sales AMER",    "Sales Workspace",     "Sales AMER Model",   'EVALUATE ROW("value", [Total Revenue])',         "ONCE_PER_DAY"),
    ("Total Sales",     "Sales EMEA",    "Sales Workspace",     "Sales EMEA Model",   'EVALUATE ROW("value", [Net Sales])',             "ONCE_PER_DAY"),

    ("Total Customers", "Finance",       "Finance Workspace",   "Finance Model",      'EVALUATE ROW("value", [Customer Count])',        "ONCE_PER_DAY"),
    ("Total Customers", "Sales AMER",    "Sales Workspace",     "Sales AMER Model",   'EVALUATE ROW("value", [Distinct Customers])',   "ONCE_PER_DAY"),
]
# ─────────────────────────────────────────────────────────────────────────────

from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([
    StructField("check_name",     StringType()),
    StructField("model_name",     StringType()),
    StructField("workspace_name", StringType()),
    StructField("dataset_name",   StringType()),
    StructField("dax_expression", StringType()),
    StructField("run_frequency",  StringType()),
])

df = spark.createDataFrame(checks, schema=schema).withColumn("is_active", lit(True))

# Upsert — safe to re-run, won't create duplicates
df.createOrReplaceTempView("_new_checks")
spark.sql(f"""
    MERGE INTO {FULL_SCHEMA}.check_registry AS t
    USING _new_checks AS s
      ON t.check_name = s.check_name AND t.model_name = s.model_name
    WHEN NOT MATCHED THEN INSERT *
    WHEN MATCHED THEN UPDATE SET 
      t.run_frequency = s.run_frequency,
      t.dax_expression = s.dax_expression
""")

count = spark.sql(f"SELECT COUNT(*) FROM {FULL_SCHEMA}.check_registry WHERE is_active = true").collect()[0][0]
print(f"✓  check_registry now has {count} active row(s)")

## View Your Checks

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT check_name, model_name, workspace_name, dataset_name, run_frequency 
FROM {FULL_SCHEMA}.check_registry 
WHERE is_active = true
ORDER BY check_name, model_name
""").toPandas())

print(f"\n✓  Ready to run the validation job")